# 06 — Exploratory structural analysis of Dutch PEP coverage

This notebook produces the supplementary tables used to interpret category-level coverage patterns.

It does **not** estimate a causal effect or test a statistical correlation. The number of taxonomy categories is small and benchmark records are not independent because individuals may hold multiple roles. The notebook therefore reports descriptive structural patterns relating to:

- category grouping;
- official-name representation;
- role level within party governance; and
- dataset-membership evidence for matched records.

Dataset membership is used only as provenance evidence for why a person may appear in the OpenSanctions export. It does **not** prove that the exact current benchmark role is encoded in a source dataset.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve the repository root whether this notebook is run from /notebooks or the project root.
CURRENT_DIR = Path.cwd().resolve()
PROJECT_DIR = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR

OUTPUT_DIR = PROJECT_DIR / "data" / "output"
MATCH_RESULTS_PATH = OUTPUT_DIR / "benchmark_v3_opensanctions_match_results.csv"

if not MATCH_RESULTS_PATH.exists():
    raise FileNotFoundError(
        "Run 03_match_opensanctions.ipynb first. Expected file not found: "
        f"{MATCH_RESULTS_PATH}"
    )

match_df = pd.read_csv(MATCH_RESULTS_PATH)
print(f"Loaded {len(match_df):,} benchmark records from {MATCH_RESULTS_PATH.name}.")


In [ ]:
# -------------------------------------------------------------------
# Helper functions and common indicators
# -------------------------------------------------------------------

def as_bool(series: pd.Series) -> pd.Series:
    """Convert boolean-like CSV values safely to True/False."""
    return series.astype(str).str.strip().str.lower().eq("true")

for column in [
    "strict_confirmed_match",
    "rule_based_probable_match",
    "included_in_extended_coverage",
]:
    match_df[column] = as_bool(match_df[column])

match_df["has_parentheses"] = match_df["person_name"].fillna("").str.contains(r"\(", regex=True)
match_df["starts_with_initials"] = match_df["person_name"].fillna("").str.match(r"^(?:[A-Z]\.){1,}")

category_group_map = {
    "head_of_state": "Constitutional and parliamentary",
    "national_executive": "Constitutional and parliamentary",
    "house_members": "Constitutional and parliamentary",
    "senate_members": "Constitutional and parliamentary",
    "high_judiciary_rvs": "High judicial bodies",
    "high_judiciary_hr": "High judicial bodies",
    "high_judiciary_cbb": "High judicial bodies",
    "high_judiciary_crvb": "High judicial bodies",
    "ambassadors": "Diplomatic service",
    "charges_d_affaires": "Diplomatic service",
    "registered_party_board": "Political-party governance",
    "court_of_audit": "Other institutional and security roles",
    "central_bank_board": "Other institutional and security roles",
    "senior_military": "Other institutional and security roles",
}

match_df["structural_group"] = match_df["taxonomy_category"].map(category_group_map)
if match_df["structural_group"].isna().any():
    missing = match_df.loc[match_df["structural_group"].isna(), "taxonomy_category"].unique()
    raise ValueError(f"Unmapped taxonomy categories: {missing}")


In [ ]:
# -------------------------------------------------------------------
# Table 1: Structural coverage groups
# -------------------------------------------------------------------

structural_group_summary = (
    match_df.groupby("structural_group", as_index=False)
    .agg(
        benchmark_records=("benchmark_record_id", "size"),
        strict_matches=("strict_confirmed_match", "sum"),
        initial_surname_matches=("rule_based_probable_match", "sum"),
        extended_matches=("included_in_extended_coverage", "sum"),
        ambiguous_exact_name_candidates=(
            "final_match_status",
            lambda values: (values == "ambiguous_exact_name_candidate").sum(),
        ),
        not_identified=(
            "final_match_status",
            lambda values: (values == "not_identified_by_implemented_matching").sum(),
        ),
    )
)

structural_group_summary["strict_coverage_pct"] = (
    100 * structural_group_summary["strict_matches"]
    / structural_group_summary["benchmark_records"]
).round(1)

structural_group_summary["extended_coverage_pct"] = (
    100 * structural_group_summary["extended_matches"]
    / structural_group_summary["benchmark_records"]
).round(1)

group_order = [
    "Constitutional and parliamentary",
    "High judicial bodies",
    "Diplomatic service",
    "Political-party governance",
    "Other institutional and security roles",
]

structural_group_summary["sort_order"] = structural_group_summary["structural_group"].map(
    {group: position for position, group in enumerate(group_order)}
)
structural_group_summary = (
    structural_group_summary.sort_values("sort_order")
    .drop(columns="sort_order")
    .reset_index(drop=True)
)

display(structural_group_summary)
structural_group_summary.to_csv(
    OUTPUT_DIR / "thesis_table_structural_coverage_groups.csv",
    index=False,
)


In [ ]:
# -------------------------------------------------------------------
# Table 2: Party-governance role gradient
# -------------------------------------------------------------------

party_df = match_df.loc[match_df["taxonomy_category"] == "registered_party_board"].copy()
party_role = party_df["role_title"].fillna("").str.lower()

party_df["party_role_group"] = np.select(
    [
        party_role.str.contains("voorzitter", regex=False),
        party_role.str.contains("penningmeester", regex=False),
        party_role.str.contains("secretaris", regex=False),
    ],
    [
        "Chair or vice-chair",
        "Treasurer",
        "Secretary",
    ],
    default="Other board role",
)

party_role_summary = (
    party_df.groupby("party_role_group", as_index=False)
    .agg(
        benchmark_records=("benchmark_record_id", "size"),
        strict_matches=("strict_confirmed_match", "sum"),
        ambiguous_exact_name_candidates=(
            "final_match_status",
            lambda values: (values == "ambiguous_exact_name_candidate").sum(),
        ),
        not_identified=(
            "final_match_status",
            lambda values: (values == "not_identified_by_implemented_matching").sum(),
        ),
    )
)

party_role_summary["strict_coverage_pct"] = (
    100 * party_role_summary["strict_matches"]
    / party_role_summary["benchmark_records"]
).round(1)

party_order = ["Chair or vice-chair", "Treasurer", "Secretary", "Other board role"]
party_role_summary["sort_order"] = party_role_summary["party_role_group"].map(
    {group: position for position, group in enumerate(party_order)}
)
party_role_summary = (
    party_role_summary.sort_values("sort_order")
    .drop(columns="sort_order")
    .reset_index(drop=True)
)

display(party_role_summary)
party_role_summary.to_csv(
    OUTPUT_DIR / "thesis_table_party_governance_role_coverage.csv",
    index=False,
)


In [ ]:
# -------------------------------------------------------------------
# Table 3: Judicial name-representation indicators
# -------------------------------------------------------------------

judicial_categories = [
    "high_judiciary_rvs",
    "high_judiciary_hr",
    "high_judiciary_cbb",
    "high_judiciary_crvb",
]

judicial_name_summary = (
    match_df.loc[match_df["taxonomy_category"].isin(judicial_categories)]
    .groupby("taxonomy_category", as_index=False)
    .agg(
        benchmark_records=("benchmark_record_id", "size"),
        names_with_parentheses=("has_parentheses", "sum"),
        names_starting_with_initials=("starts_with_initials", "sum"),
        strict_matches=("strict_confirmed_match", "sum"),
        initial_surname_matches=("rule_based_probable_match", "sum"),
        not_identified=(
            "final_match_status",
            lambda values: (values == "not_identified_by_implemented_matching").sum(),
        ),
    )
)

display(judicial_name_summary)
judicial_name_summary.to_csv(
    OUTPUT_DIR / "thesis_table_judicial_name_representation.csv",
    index=False,
)


In [ ]:
# -------------------------------------------------------------------
# Table 4: Source-path evidence among confirmed matches
# -------------------------------------------------------------------

def count_membership(category: str, membership_label: str) -> dict:
    subset = match_df.loc[
        (match_df["taxonomy_category"] == category)
        & (match_df["strict_confirmed_match"])
    ].copy()
    membership_count = subset["dataset"].fillna("").str.contains(
        membership_label,
        regex=False,
    ).sum()

    return {
        "category": category,
        "strict_matches": len(subset),
        "strict_matches_with_membership": int(membership_count),
        "membership_share_pct": round(
            100 * membership_count / len(subset),
            1,
        ) if len(subset) else 0.0,
        "membership_label": membership_label,
    }

source_path_summary = pd.DataFrame(
    [
        count_membership(
            "house_members",
            "Netherlands House of Representatives",
        ),
        count_membership(
            "senate_members",
            "Netherlands Senate",
        ),
    ]
)

display(source_path_summary)
source_path_summary.to_csv(
    OUTPUT_DIR / "thesis_table_direct_source_path_evidence.csv",
    index=False,
)

print("Structural-analysis tables written to data/output/.")
